In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
df = pd.read_csv('.\data\premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1333 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [48]:
#bmi의 널처리
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [49]:
from sklearn.preprocessing import LabelEncoder

In [50]:
# 문자열 데이터의 수치화 > LabelEncoder
col_list = ['sex', 'smoker', 'region']
for col in col_list:
  enc = LabelEncoder()
  df[col] = enc.fit_transform(df[col])

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [51]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [52]:
# 스케일링 하기

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, random_state=42)

In [53]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [54]:
# 연속형 컬럼
num_cols = ['bmi']

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

display(X_train[num_cols].describe().round(4))

,bmi
count,1070.0000
mean,0.0000
std,1.0005
min,-2.4246
25%,-0.7201
50%,-0.0506
75%,0.6439
max,3.7506


In [61]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [62]:
# 선형회귀 모델
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae, mse, rmse, r2

(4172.783193083617,
 33640928.3211508,
 np.float64(5800.0800271333155),
 0.7833094802263592)

# 다항회귀

In [63]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [70]:
degree = [2, 3, 4]

for deg in degree:
  model_poly = Pipeline([('poly', PolynomialFeatures(degree=deg, include_bias=False)),
                         ('linear', LinearRegression())])
  model_poly.fit(X_train, y_train)
  poly_pred = model_poly.predict(X_test)

  mae = mean_absolute_error(y_test, poly_pred)
  mse = mean_squared_error(y_test, poly_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, poly_pred)
mae, mse, rmse, r2

(3120.6620650772716,
 34379524.83832539,
 np.float64(5863.405566590579),
 0.778551975864955)

In [66]:
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state=0)
model_rf.fit(X_train,y_train)
y_pred_rf = model_rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)
mae, mse, rmse, r2

(2499.0538510119104,
 21474275.29598436,
 np.float64(4634.034451316084),
 0.861678255985478)

In [67]:
# 특성 중요도
import pandas as pd
feature_names = ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
importance_df = pd.DataFrame({
  'feature': feature_names,
  'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f'특성중요도 \n-------------------------------------\n {importance_df}')

특성중요도 
-------------------------------------
     feature  importance
4    smoker    0.610743
2       bmi    0.207945
0       age    0.136268
3  children    0.022684
5    region    0.015433
1       sex    0.006929


# XGBRegressor

In [68]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 모델 생성
model_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=0,
    objective='reg:squarederror'
)

# 학습
model_xgb.fit(X_train, y_train)

# 예측
y_pred_xgb = model_xgb.predict(X_test)

# 평가
mae = mean_absolute_error(y_test, y_pred_xgb)
mse = mean_squared_error(y_test, y_pred_xgb)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_xgb)

mae, mse, rmse, r2

(2530.7423597910442,
 20312524.064407628,
 np.float64(4506.941764035522),
 0.8691614168487813)

In [69]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'random_state': 0
    }
    model = RandomForestRegressor(**params)
    score = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f'Best MAE : {round(study.best_value, 4)}')
print(f'Best Params : {study.best_params}')

[I 2026-02-27 17:25:42,094] A new study created in memory with name: no-name-7ffad9cd-b74a-47c9-9924-9bbb46b252b6
[I 2026-02-27 17:25:43,797] Trial 0 finished with value: 2571.045056871232 and parameters: {'n_estimators': 300, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 1}. Best is trial 0 with value: 2571.045056871232.
[I 2026-02-27 17:25:44,234] Trial 1 finished with value: 2560.8058359901993 and parameters: {'n_estimators': 60, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 3}. Best is trial 1 with value: 2560.8058359901993.
[I 2026-02-27 17:25:45,335] Trial 2 finished with value: 2552.2630387809013 and parameters: {'n_estimators': 212, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 1}. Best is trial 2 with value: 2552.2630387809013.
[I 2026-02-27 17:25:46,764] Trial 3 finished with value: 2527.1802514899746 and parameters: {'n_estimators': 247, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 2}. Best is trial 3 with value: 2527

Best MAE : 2504.415
Best Params : {'n_estimators': 225, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 3}
